## Christina Stuhl: Multipanel Figure

In [1]:
# Dependencies
# This program builds a 4-panel figure summarizing the extraction of dissolved gas concentrations
# from an in-situ Residual Gas Analyzer (RGA) workflow.
# The panels include:
# a) a single RGA spectrum with selected gas peaks marked by amu
# b) a table of gas constants and atomic masses
# c) a 2017 dissolved gas time series derived from Henry's Law
# d) a verification plot showing that concentration scales linearly with partial pressure

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D

In [2]:
# SETTINGS
# Main figure title and figure caption
# The caption explains each panel and the overall logic of the workflow

FIG_TITLE = "Acquiring Dissolved Gas Concentrations from an In-Situ Residual Gas Analyzer (RGA)"
FIG_CAPTION = (
    "RGA Data Acquisition: A comprehensive overview of the extraction methods for the Residual Gas "
    "Analyzer (RGA) at Southern Hydrate Ridge Summit A.\n"
    "Each RGA .txt file included a spectra of values associated with the partial pressure (Torr) of a "
    "specific gas identified by an atomic mass (amu).\n"
    "a) Features a single .txt file from one measurement taken by the RGA in 2017, and includes markers "
    "identifying the gases selected for this study.\n"
    "b) Includes the Henry's law constant values k that were used to convert each partial pressure into a concentration that could be understood.\n"
    "c) A time series of 2017 without interpolation, i.e. a NaN value does not equal zero so when the instrument is not active no data is displayed.\n"
    "d) A check verifying that the calculated values obey Henry's law. Each trend obeys the law C = kP, such that k is always the Henry's law constant provided in table b.\n"
    "This concludes the process by which concentration data was collected from the RGA."
)

# Figure size and relative panel widths
FIGSIZE = (18, 8.5)

LEFT_COL_WIDTH = 2.5
RIGHT_COL_WIDTH = 1.5

# Grid spacing between columns and rows
COLUMN_SPACING = 0.25
ROW_SPACING = 0.45

# Border appearance for each panel
BORDER_COLOR = "black"
BORDER_WIDTH = 1.2

# File path for the single RGA spectrum used in Panel A
RGA_FILE = r"C:\Users\chero\Downloads\09-22-16-rga.txt"

# Time binning to prevent huge arrays (pick one)
# These settings are retained here as part of the workflow configuration
# and describe the intended time aggregation / gap handling scale
TIME_BIN = "1H"  # "15min", "1H", "6H", "1D" (daily)
GAP_FOR_BREAK = "2H"  # gap threshold for breaking lines (match TIME_BIN scale)

# MASSPA time series file for Panel B + D
# This file contains the 2017 time series of partial pressures used to compute concentrations
CSV_FILE = r"C:\Users\chero\OneDrive\Desktop\W2026\ESS 469\MASSPA_2017_Time_Series_Ready.csv"

In [3]:
# Axis limits for Panel D (plot d)
# These limits control the viewing window for the Henry's Law validation plot

PANEL_D_XLIM = (0, 0.1e-8)
PANEL_D_YLIM = (0, 0.2e-11)

# Layout tweaks
# These values adjust subplot spacing and the final caption placement below the figure

ROW1_ROW2_HSPACE = 0.55
CAPTION_Y = 0.02
CAPTION_X = 0.05
CAPTION_LINE_SPACING = 1.2

In [4]:
# GAS LIST
# Each gas dictionary stores:
# - the full gas name
# - the plotted chemical formula
# - the target amu used to extract the partial pressure signal
# - the Henry's Law constant
# - the unit used for the time series panel
# - the axis assignment for the time series panel
# - the display color used consistently across all panels

GASES = [
    dict(name="Hydrogen", formula="H$_2$", amu=2, henry=7.8e-4, ts_unit="uM", ts_axis="left", color="#5CBCE6",
         stats_extra=r"($C_{H_2} = k_{H_2} P_{H_2}$)"),
    dict(name="Methane", formula="CH$_4$", amu=16, henry=1.4e-3, ts_unit="uM", ts_axis="left", color="#6BE88E",
         stats_extra=r"($C_{CH_4} = k_{CH_4} P_{CH_4}$)"),
    dict(name="Hydrogen sulfide", formula="H$_2$S", amu=34, henry=1.0e-1, ts_unit="mM", ts_axis="right",
         color="#EBDF78", stats_extra=r"($C_{H_2S} = k_{H_2S} P_{H_2S}$)"),
    dict(name="Carbon dioxide", formula="CO$_2$", amu=44, henry=1.7e-3, ts_unit="uM", ts_axis="left", color="#B439E1",
         stats_extra=r"($C_{CO_2} = k_{CO_2} P_{CO_2}$)"),
    dict(name="Oxygen", formula="O$_2$", amu=32, henry=1.3e-3, ts_unit="uM", ts_axis="left", color="#4D4BE3",
         stats_extra=r"($C_{O_2} = k_{O_2} P_{O_2}$)"),
    dict(name="Nitrogen", formula="N$_2$", amu=28, henry=6.5e-4, ts_unit="uM", ts_axis="left", color="#E153AA",
         stats_extra=r"($C_{N_2} = k_{N_2} P_{N_2}$)"),
]

# Pressure units for the raw partial pressure data
# These are used later when converting to atm for the Henry's Law check
PRESSURE_UNITS = "torr"

In [5]:
# PARSE RGA FILE (for Panel A)
# Read the single RGA text file and extract:
# - amu values
# - partial pressure values in torr
# Then convert the partial pressure values from torr to atm for plotting

amu = []
pressure_torr = []

with open(RGA_FILE, "r") as f:
    for line in f:
        line = line.strip()
        if "," in line:
            parts = line.split(",")
            if len(parts) >= 2:
                try:
                    m = float(parts[0])
                    p = float(parts[1])
                    amu.append(m)
                    pressure_torr.append(p)
                except ValueError:
                    pass

amu = np.array(amu)
pressure_atm = np.array(pressure_torr) / 760.0

In [6]:
# LOAD MASSPA (for Panels B + D)
# Load the time series CSV file, coerce the important columns to numeric/datetime formats,
# remove invalid rows, and isolate only data from calendar year 2017

df = pd.read_csv(CSV_FILE)

df["Timestamp"] = pd.to_datetime(df["Timestamp"])
df["Mass"] = pd.to_numeric(df["Mass"], errors="coerce")
df["Partial_Pressure"] = pd.to_numeric(df["Partial_Pressure"], errors="coerce")

df = df.dropna(subset=["Timestamp", "Mass", "Partial_Pressure"])
df_2017 = df[(df["Timestamp"] >= "2017-01-01") & (df["Timestamp"] < "2018-01-01")].copy()

MemoryError: Unable to allocate 225. MiB for an array with shape (29502070,) and data type int64

In [ ]:
# PANEL B: compute concentrations over 2017
# For each target gas:
# - sum the total pressure at each timestamp
# - isolate the pressure contribution from the target amu
# - compute dissolved concentration using Henry's Law
# - return the result as a timestamped concentration time series in uM

def compute_concentration_uM(df_in, target_amu, henry_const):
    out = df_in.groupby("Timestamp").agg(
        Total_Pressure=("Partial_Pressure", "sum"),
        Gas_Pressure=("Partial_Pressure",
                      lambda x: df_in.loc[x.index, "Partial_Pressure"]
                      [df_in.loc[x.index, "Mass"].round(6) == target_amu].sum())
    ).reset_index()

    out["Concentration_nM"] = (out["Gas_Pressure"] / out["Total_Pressure"]) * henry_const * 1e9
    out["Concentration_uM"] = out["Concentration_nM"] / 1000.0
    return out[["Timestamp", "Concentration_uM"]]


# Break the plotted line anywhere there is a large time gap
# This keeps missing instrument intervals from being plotted as continuous data
def break_gaps(df_in, time_col="Timestamp", value_col="Concentration_uM", gap=pd.Timedelta("1D")):
    df2 = df_in.sort_values(time_col).reset_index(drop=True).copy()
    times = df2[time_col].to_list()
    vals = df2[value_col].to_list()

    out_t = []
    out_v = []

    for i in range(len(times)):
        out_t.append(times[i])
        out_v.append(vals[i])

        if i < len(times) - 1:
            if (times[i + 1] - times[i]) > gap:
                out_t.append(times[i] + pd.Timedelta(seconds=1))
                out_v.append(np.nan)

    return np.array(out_t, dtype="datetime64[ns]"), np.array(out_v, dtype=float)


# Build the final time series dictionary used by Panel B
# Each entry stores the time array, concentration array, plotting unit,
# axis assignment, color, and gas metadata
gas_ts = {}
for g in GASES:
    ts_uM = compute_concentration_uM(df_2017, g["amu"], g["henry"]).copy()

    if g["ts_unit"].lower() == "mm":
        ts_uM["Concentration_mM"] = ts_uM["Concentration_uM"] / 1000.0
        t_arr, c_arr = break_gaps(ts_uM, value_col="Concentration_mM", gap=pd.Timedelta("1D"))
        ylab_unit = "mM"
    elif g["ts_unit"].lower() == "um":
        t_arr, c_arr = break_gaps(ts_uM, value_col="Concentration_uM", gap=pd.Timedelta("1D"))
        ylab_unit = "µM"
    else:
        raise ValueError("ts_unit must be 'uM' or 'mM' for each gas in GASES.")

    gas_ts[g["formula"]] = dict(
        t=t_arr, c=c_arr, unit=ylab_unit,
        axis=g["ts_axis"].lower(),
        color=g["color"],
        amu=g["amu"],
        name=g["name"],
        henry=g["henry"],
        formula=g["formula"]
    )

In [ ]:
# PANEL D: Checking Henry's Law
# Convert partial pressure data to atm when needed
# This ensures the Henry's Law validation is done in consistent physical units

def to_atm(p):
    if PRESSURE_UNITS.lower() == "torr":
        return p / 760.0
    if PRESSURE_UNITS.lower() == "atm":
        return p
    raise ValueError("PRESSURE_UNITS must be 'torr' or 'atm'.")


# Build a time series of gas-specific partial pressure for a selected amu
# and convert that gas pressure into atm
def gas_pressure_series(df_in: pd.DataFrame, target_amu: int) -> pd.DataFrame:
    df_tmp = df_in.copy()
    df_tmp["Mass_rounded"] = df_tmp["Mass"].round(6)

    out = df_tmp.groupby("Timestamp").agg(
        Gas_Pressure=("Partial_Pressure",
                      lambda s: df_tmp.loc[s.index, "Partial_Pressure"]
                      [df_tmp.loc[s.index, "Mass_rounded"] == target_amu].sum())
    ).reset_index()

    out["P_gas_atm"] = to_atm(out["Gas_Pressure"])
    return out[["Timestamp", "P_gas_atm"]]


# Fit a straight line and compute R^2 for the Henry's Law comparison
# This is used to show how closely the calculated concentrations follow C = kP
def fit_line_and_r2(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    if len(x) < 2:
        return np.nan, np.nan, np.nan

    m, b = np.polyfit(x, y, 1)
    y_pred = m * x + b
    ss_res = np.sum((y - y_pred) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)
    r2 = 1.0 - ss_res / ss_tot if ss_tot != 0 else np.nan
    return m, b, r2


# Build a Henry's Law validation dictionary for each gas
# Each gas stores:
# - the gas-only pressure series in atm
# - the predicted concentration in mol/L
# - the fitted line slope/intercept
# - the coefficient of determination
gas_henry = {}
for g in GASES:
    gp = gas_pressure_series(df_2017, g["amu"])
    gp["C_mol_L"] = g["henry"] * gp["P_gas_atm"]
    gp = gp[np.isfinite(gp["C_mol_L"]) & np.isfinite(gp["P_gas_atm"])].copy()

    k, b, r2 = fit_line_and_r2(gp["P_gas_atm"], gp["C_mol_L"])
    gas_henry[g["formula"]] = dict(df=gp, k=k, b=b, r2=r2, **g)

In [ ]:
# FIGURE + GRID
# Create the 2 x 2 panel figure and assign one axis to each panel:
# A = RGA spectrum
# B = dissolved gas time series
# C = gas constants table
# D = Henry's Law check

fig = plt.figure(figsize=FIGSIZE)

outer_gs = GridSpec(
    2, 2,
    width_ratios=[LEFT_COL_WIDTH, RIGHT_COL_WIDTH],
    height_ratios=[1, 1],
    wspace=COLUMN_SPACING,
    hspace=ROW_SPACING,
    figure=fig
)

ax_A = fig.add_subplot(outer_gs[0, 0])
ax_B = fig.add_subplot(outer_gs[1, 0])
ax_C = fig.add_subplot(outer_gs[0, 1])
ax_D = fig.add_subplot(outer_gs[1, 1])

# Increased bottom margin to create more space above the caption
fig.subplots_adjust(left=0.05, right=0.985, bottom=0.26, top=0.90, hspace=ROW1_ROW2_HSPACE)

In [ ]:
# FUNCTION add panel border
# Apply a consistent visible border to every subplot panel

def add_panel_border(ax, lw=BORDER_WIDTH):
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(lw)
        spine.set_color(BORDER_COLOR)

In [ ]:
# PANEL A — RGA SPECTRUM
# Plot the single RGA mass spectrum and mark the selected gas peaks with colored dashed lines

ax_A.set_title("a) RGA Spectra Peaks", fontweight="bold")
ax_A.plot(amu, pressure_atm, color="gray")

for g in GASES:
    ax_A.axvline(
        g["amu"],
        color=g["color"],
        linestyle="--",
        linewidth=1.5,
        label=f'{g["amu"]} amu ({g["formula"]})'
    )

ax_A.set_xlabel("Mass (amu)")
ax_A.set_ylabel("Partial Pressure (atm)")

xmin = np.floor(amu.min())
xmax = np.ceil(amu.max())
ax_A.set_xticks(np.arange(xmin, xmax + 10, 10))

ax_A.set_yscale("symlog", linthresh=1e-11)
ax_A.set_ylim(0, 1e-7)

ax_A.legend(loc="best")
add_panel_border(ax_A)

In [ ]:
# PANEL C — table
# Create a compact reference table showing:
# - chemical formula
# - atomic mass in amu
# - Henry's Law constant in M/atm

ax_C.set_title("b) Table of Values", fontweight="bold")
ax_C.set_xticks([])
ax_C.set_yticks([])

table_data = [
    ["UNITS", "amu", "M/atm"],
    ["Formula", "Atomic Mass", "k$_H$"],
]

for g in GASES:
    table_data.append([g["formula"], f'{g["amu"]:.2f}', f'{g["henry"]:.1e}'])

col_widths = [0.25, 0.40, 0.35]

tbl = ax_C.table(
    cellText=table_data,
    cellLoc="center",
    loc="center",
    bbox=[0, 0, 1, 1],
    colWidths=col_widths
)

for cell in tbl.get_celld().values():
    cell.get_text().set_wrap(True)

tbl.auto_set_font_size(False)
tbl.set_fontsize(13)

for c in range(len(table_data[0])):
    tbl[(0, c)].get_text().set_weight("bold")
for c in range(len(table_data[1])):
    tbl[(1, c)].get_text().set_weight("bold")

add_panel_border(ax_C)

In [ ]:
# FIGURE TITLE + WRAPPED CAPTION
# Add the main figure title at the top and the wrapped explanatory caption below the panels

fig.suptitle(FIG_TITLE, x=0.05, ha="left", fontsize=14, fontweight="bold")

fig.text(
    CAPTION_X,
    CAPTION_Y,
    FIG_CAPTION,
    ha="left",
    va="bottom",
    fontsize=12,
    wrap=True,
    linespacing=CAPTION_LINE_SPACING
)

plt.show()